In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from datetime import datetime
import hashlib


MATRIX_PRESET = "cartesian"
USE_USPLAT = False
DROPOUT_DEFAULT = "no_dropout"


FINAL_COLUMNS = [
    "status",
    "run_hash",
    "scene_name",
    "variant_name",
    "variant_index",
    "matrix_preset",
    "isotropy",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
    "use_usplat",
    "config_path",
    "generated_config_path",
    "model_path",
    "checkpoint_index",
    "checkpoint_filename",
    "checkpoint_path",
    "checkpoint_name_iteration",
    "eval_requested_split",
    "eval_split_used",
    "eval_checkpoint_iteration",
    "psnr",
    "ssim",
    "lpips",
    "render_fps",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_bytes",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
    "metrics_jsonl_path",
    "metrics_csv_path",
    "created_at",
]


def stable_hash(text: str, length: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:length]


def parse_variant_dirname(dirname: str) -> dict:
    """
    Supports:

    5-part:
        isotropic--rgb--sort--no_pruning--no_ess

    6-part:
        isotropic--rgb--sort--no_pruning--no_dropout--no_ess

    7-part:
        anisotropic--rgb--sort--interleaved_prune_densify--ess--dropout--no_usplat
    """
    parts = dirname.split("--")

    use_usplat = USE_USPLAT

    if len(parts) == 5:
        isotropy, appearance, sorting, pruning, ess = parts
        dropout = DROPOUT_DEFAULT

    elif len(parts) == 6:
        isotropy, appearance, sorting, pruning, dropout, ess = parts

    elif len(parts) == 7:
        isotropy, appearance, sorting, pruning, ess, dropout, usplat = parts
        use_usplat = usplat != "no_usplat"

    else:
        raise ValueError(f"Unexpected variant folder name: {dirname}")

    variant_parts = [
        isotropy,
        appearance,
        sorting,
        pruning,
        dropout,
        ess,
    ]

    return {
        "isotropy": isotropy,
        "appearance": appearance,
        "sorting": sorting,
        "pruning": pruning,
        "dropout": dropout,
        "ess": ess,
        "use_usplat": use_usplat,
        "variant_name": "__".join(variant_parts),
    }

def value_or(record: dict, key: str, fallback):
    value = record.get(key, pd.NA)
    return fallback if pd.isna(value) or value == "" else value


def read_dataset_results(dataset_dir: Path, write_csv: bool = True) -> pd.DataFrame:
    dataset_dir = Path(dataset_dir)
    scene_name = dataset_dir.name
    config_path = f"configs/dnerf_ablation/{scene_name}.yaml"
    output_csv = dataset_dir.parent / f"{scene_name}_checkpoint_eval_metrics_all.csv"

    rows = []
    created_at = datetime.now().replace(microsecond=0).isoformat()

    variant_dirs = sorted(p for p in dataset_dir.iterdir() if p.is_dir())

    for variant_index, variant_dir in enumerate(variant_dirs):
        metrics_csv = variant_dir / "checkpoint_eval_metrics.csv"

        if not metrics_csv.exists():
            print(f"Skipping, no metrics file: {metrics_csv}")
            continue

        parsed = parse_variant_dirname(variant_dir.name)

        df = pd.read_csv(metrics_csv, skipinitialspace=True)
        df.columns = df.columns.str.strip()

        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
                df[col] = df[col].astype("string").str.strip()

        for _, row in df.iterrows():
            record = row.to_dict()

            record["run_hash"] = value_or(
                record,
                "run_hash",
                stable_hash(str(variant_dir.resolve())),
            )

            record["scene_name"] = value_or(
                record,
                "scene_name",
                scene_name,
            )

            record["variant_name"] = value_or(
                record,
                "variant_name",
                parsed["variant_name"],
            )

            record["variant_index"] = variant_index
            record["matrix_preset"] = MATRIX_PRESET

            for key in ["isotropy", "appearance", "sorting", "pruning", "dropout", "ess"]:
                record[key] = value_or(record, key, parsed[key])

            record["use_usplat"] = USE_USPLAT
            record["config_path"] = config_path

            record["model_path"] = value_or(
                record,
                "model_path",
                str(variant_dir.resolve()),
            )

            record["metrics_csv_path"] = str(metrics_csv.resolve())
            record["metrics_jsonl_path"] = str(
                (variant_dir / "checkpoint_eval_metrics.jsonl").resolve()
            )

            record["training_wall_clock_sec_at_checkpoint"] = value_or(
                record,
                "training_wall_clock_sec_at_checkpoint",
                pd.NA,
            )

            # chkpnt_best.pth may have no number in filename, so use loaded eval iteration.
            record["checkpoint_name_iteration"] = value_or(
                record,
                "checkpoint_name_iteration",
                record.get("eval_checkpoint_iteration", pd.NA),
            )

            record["created_at"] = created_at
            rows.append(record)

    combined = pd.DataFrame(rows)

    for col in FINAL_COLUMNS:
        if col not in combined.columns:
            combined[col] = pd.NA

    byte_cols = [
        c for c in combined.columns
        if c.endswith("_bytes") or "bytes" in c.lower()
    ]

    for col in byte_cols:
        mb_col = col.replace("_bytes", "_mb")
        combined[mb_col] = pd.to_numeric(combined[col], errors="coerce") / (1024 ** 2)

    for col in FINAL_COLUMNS:
        if col not in combined.columns:
            combined[col] = pd.NA

    final_df = combined[FINAL_COLUMNS].copy()

    if write_csv:
        final_df.to_csv(output_csv, index=False)
        print(f"Wrote {len(final_df)} rows to {output_csv}")

    return final_df


def read_all_dataset_results(output_root: Path = Path("output")) -> dict[str, pd.DataFrame]:
    results = {}

    for ablation_dir in sorted(output_root.glob("ablations_*")):
        if not ablation_dir.is_dir():
            continue

        dataset_dirs = sorted(
            p for p in ablation_dir.iterdir()
            if p.is_dir() and p.name != "generated_configs"
        )

        for dataset_dir in dataset_dirs:
            key = f"{ablation_dir.name}/{dataset_dir.name}"
            results[key] = read_dataset_results(dataset_dir)

    return results

dataset_results = read_all_dataset_results()
dataset_results["ablations_usplat_sort_dropout_off/trex"]["use_usplat"] = True
dataset_results["ablations_no_usplat_dropout_on/trex"]["dropout"] = "yes_dropout"

def infer_use_usplat(row):
    tokens = []

    if pd.notna(row.get("variant_name")):
        tokens += str(row["variant_name"]).split("__")

    if pd.notna(row.get("model_path")):
        tokens += Path(str(row["model_path"])).name.split("--")

    if pd.notna(row.get("generated_config_path")):
        tokens += Path(str(row["generated_config_path"])).stem.split("__")

    return "use_usplat" in tokens

dataset_results["ablations_bouncing_balls_usplat_vs_no_usplat_7k/bouncingballs"]["use_usplat"] = dataset_results["ablations_bouncing_balls_usplat_vs_no_usplat_7k/bouncingballs"].apply(infer_use_usplat, axis=1)


def merge_dataset_results(
    dataset_results: dict[str, pd.DataFrame],
    drop_rules: list[tuple[str, str, object]] | None = None,
) -> pd.DataFrame:
    drop_rules = drop_rules or []
    rules_by_key = {}

    for key, col, value in drop_rules:
        rules_by_key.setdefault(key, []).append((col, value))

    merged = []

    for key, df in dataset_results.items():
        df = df.copy()
        original_len = len(df)

        drop_mask = pd.Series(False, index=df.index)

        for col, value in rules_by_key.get(key, []):
            if col not in df.columns:
                print(f"{key}: skip missing drop column: {col}")
                continue

            drop_mask |= df[col].astype(str).str.lower().eq(str(value).lower())

        dropped = int(drop_mask.sum())
        kept = df.loc[~drop_mask].copy()

        frac = dropped / original_len if original_len else 0.0
        print(f"{key}: dropped {dropped}/{original_len} rows ({frac:.1%})")

        kept.insert(0, "dataset_key", key)
        merged.append(kept)

    final_df = pd.concat(merged, ignore_index=True) if merged else pd.DataFrame()
    return final_df


final_df = merge_dataset_results(
    dataset_results,
    drop_rules=[
        ("ablations_no_usplat_dropout_off/trex", "sorting", "sort_free"),
    ],
)

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import hashlib

import pandas as pd
import numpy as np


MATRIX_PRESET = "cartesian"
USE_USPLAT_DEFAULT = False
DROPOUT_DEFAULT = "no_dropout"


FINAL_COLUMNS = [
    "status",
    "run_hash",
    "scene_name",
    "variant_name",
    "variant_index",
    "matrix_preset",
    "isotropy",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
    "use_usplat",
    "config_path",
    "generated_config_path",
    "model_path",
    "checkpoint_index",
    "checkpoint_filename",
    "checkpoint_path",
    "checkpoint_name_iteration",
    "eval_requested_split",
    "eval_split_used",
    "eval_checkpoint_iteration",
    "psnr",
    "ssim",
    "lpips",
    "render_fps",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_bytes",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
    "metrics_jsonl_path",
    "metrics_csv_path",
    "created_at",
]


DATASET_OVERRIDES = {
    "ablations_usplat_sort_dropout_off/trex": {
        "use_usplat": True,
    },
    "ablations_no_usplat_dropout_on/trex": {
        "dropout": "yes_dropout",
    },
}

INFER_USPLAT_KEYS = {
    "ablations_bouncing_balls_usplat_vs_no_usplat_7k/bouncingballs",
}


DROP_RULES = [
    ("ablations_no_usplat_dropout_off/trex", "sorting", "sort_free"),
]


def stable_hash(text: str, length: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:length]


def is_missing(value) -> bool:
    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass

    return isinstance(value, str) and value.strip() == ""

def value_or(record: dict, key: str, fallback):
    value = record.get(key, pd.NA)
    return fallback if is_missing(value) else value


def parse_variant_dirname(dirname: str) -> dict:
    """
    Supports:

    5-part:
        isotropic--rgb--sort--no_pruning--no_ess

    6-part:
        isotropic--rgb--sort--no_pruning--no_dropout--no_ess

    7-part:
        anisotropic--rgb--sort--interleaved_prune_densify--ess--dropout--no_usplat
    """
    parts = dirname.split("--")
    use_usplat = USE_USPLAT_DEFAULT

    if len(parts) == 5:
        isotropy, appearance, sorting, pruning, ess = parts
        dropout = DROPOUT_DEFAULT

    elif len(parts) == 6:
        isotropy, appearance, sorting, pruning, dropout, ess = parts

    elif len(parts) == 7:
        isotropy, appearance, sorting, pruning, ess, dropout, usplat = parts
        use_usplat = usplat != "no_usplat"

    else:
        raise ValueError(f"Unexpected variant folder name: {dirname}")

    variant_name = "__".join(
        [isotropy, appearance, sorting, pruning, dropout, ess]
    )

    return {
        "isotropy": isotropy,
        "appearance": appearance,
        "sorting": sorting,
        "pruning": pruning,
        "dropout": dropout,
        "ess": ess,
        "use_usplat": use_usplat,
        "variant_name": variant_name,
    }


def clean_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()

    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype("string").str.strip()

    return df


def ensure_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def add_mb_columns(df: pd.DataFrame) -> pd.DataFrame:
    byte_cols = [
        col for col in df.columns
        if col.endswith("_bytes") or "bytes" in col.lower()
    ]

    for byte_col in byte_cols:
        mb_col = byte_col.replace("_bytes", "_mb")
        mb_values = pd.to_numeric(df[byte_col], errors="coerce") / (1024 ** 2)

        if mb_col in df.columns:
            df[mb_col] = df[mb_col].where(pd.notna(df[mb_col]), mb_values)
        else:
            df[mb_col] = mb_values

    return df


def build_record(
    row: pd.Series,
    *,
    variant_dir: Path,
    variant_index: int,
    parsed: dict,
    scene_name: str,
    config_path: str,
    created_at: str,
) -> dict:
    record = row.to_dict()

    defaults = {
        "run_hash": stable_hash(str(variant_dir.resolve())),
        "scene_name": scene_name,
        "variant_name": parsed["variant_name"],
        "matrix_preset": MATRIX_PRESET,
        "model_path": str(variant_dir.resolve()),
        "metrics_csv_path": str((variant_dir / "checkpoint_eval_metrics.csv").resolve()),
        "metrics_jsonl_path": str((variant_dir / "checkpoint_eval_metrics.jsonl").resolve()),
        "training_wall_clock_sec_at_checkpoint": pd.NA,
    }

    for key in ["isotropy", "appearance", "sorting", "pruning", "dropout", "ess", "use_usplat"]:
        defaults[key] = parsed[key]

    for key, fallback in defaults.items():
        record[key] = value_or(record, key, fallback)

    record["variant_index"] = variant_index
    record["config_path"] = config_path
    record["created_at"] = created_at

    # chkpnt_best.pth may have no number in filename, so use loaded eval iteration.
    record["checkpoint_name_iteration"] = value_or(
        record,
        "checkpoint_name_iteration",
        record.get("eval_checkpoint_iteration", pd.NA),
    )

    return record


def read_dataset_results(dataset_dir: Path, write_csv: bool = True) -> pd.DataFrame:
    dataset_dir = Path(dataset_dir)
    scene_name = dataset_dir.name

    config_path = f"configs/dnerf_ablation/{scene_name}.yaml"
    output_csv = dataset_dir.parent / f"{scene_name}_checkpoint_eval_metrics_all.csv"

    rows = []
    created_at = datetime.now().replace(microsecond=0).isoformat()

    variant_dirs = sorted(p for p in dataset_dir.iterdir() if p.is_dir())

    for variant_index, variant_dir in enumerate(variant_dirs):
        metrics_csv = variant_dir / "checkpoint_eval_metrics.csv"

        if not metrics_csv.exists():
            print(f"Skipping, no metrics file: {metrics_csv}")
            continue

        parsed = parse_variant_dirname(variant_dir.name)
        df = clean_string_columns(pd.read_csv(metrics_csv, skipinitialspace=True))

        for _, row in df.iterrows():
            rows.append(
                build_record(
                    row,
                    variant_dir=variant_dir,
                    variant_index=variant_index,
                    parsed=parsed,
                    scene_name=scene_name,
                    config_path=config_path,
                    created_at=created_at,
                )
            )

    combined = pd.DataFrame(rows)
    combined = ensure_columns(combined, FINAL_COLUMNS)
    combined = add_mb_columns(combined)
    combined = ensure_columns(combined, FINAL_COLUMNS)

    final_df = combined[FINAL_COLUMNS].copy()

    if write_csv:
        final_df.to_csv(output_csv, index=False)
        print(f"Wrote {len(final_df)} rows to {output_csv}")

    return final_df


def read_all_dataset_results(
    output_root: Path = Path("output"),
    write_csv: bool = True,
) -> dict[str, pd.DataFrame]:
    results = {}

    for ablation_dir in sorted(output_root.glob("ablations_*")):
        if not ablation_dir.is_dir():
            continue

        dataset_dirs = sorted(
            p for p in ablation_dir.iterdir()
            if p.is_dir() and p.name != "generated_configs"
        )

        for dataset_dir in dataset_dirs:
            key = f"{ablation_dir.name}/{dataset_dir.name}"
            results[key] = read_dataset_results(dataset_dir, write_csv=write_csv)

    return results


def infer_use_usplat(row: pd.Series) -> bool:
    tokens = []

    for col in ["variant_name", "model_path", "generated_config_path"]:
        value = row.get(col)

        if pd.isna(value):
            continue

        value = str(value)

        if col == "model_path":
            tokens.extend(Path(value).name.split("--"))
        elif col == "generated_config_path":
            tokens.extend(Path(value).stem.split("__"))
        else:
            tokens.extend(value.split("__"))

    return "use_usplat" in tokens


def apply_dataset_overrides(dataset_results: dict[str, pd.DataFrame]) -> None:
    for key, overrides in DATASET_OVERRIDES.items():
        if key not in dataset_results:
            print(f"Skipping missing override dataset: {key}")
            continue

        for col, value in overrides.items():
            dataset_results[key][col] = value

    for key in INFER_USPLAT_KEYS:
        if key not in dataset_results:
            print(f"Skipping missing inferred usplat dataset: {key}")
            continue

        dataset_results[key]["use_usplat"] = dataset_results[key].apply(
            infer_use_usplat,
            axis=1,
        )


def merge_dataset_results(
    dataset_results: dict[str, pd.DataFrame],
    drop_rules: list[tuple[str, str, object]] | None = None,
) -> pd.DataFrame:
    rules_by_key: dict[str, list[tuple[str, object]]] = {}

    for key, col, value in drop_rules or []:
        rules_by_key.setdefault(key, []).append((col, value))

    merged = []

    for key, df in dataset_results.items():
        df = df.copy()
        original_len = len(df)

        drop_mask = pd.Series(False, index=df.index)

        for col, value in rules_by_key.get(key, []):
            if col not in df.columns:
                print(f"{key}: skip missing drop column: {col}")
                continue

            drop_mask |= df[col].astype(str).str.lower().eq(str(value).lower())

        dropped = int(drop_mask.sum())
        kept = df.loc[~drop_mask].copy()

        frac = dropped / original_len if original_len else 0.0
        print(f"{key}: dropped {dropped}/{original_len} rows ({frac:.1%})")

        kept.insert(0, "dataset_key", key)
        merged.append(kept)

    return pd.concat(merged, ignore_index=True) if merged else pd.DataFrame()


ABLATION_BOOL_COLS = [
    "isotropy",
    "use_usplat",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
]

CHECKPOINT_COLS = [
    "checkpoint_index",
    "checkpoint_name_iteration",
    "eval_checkpoint_iteration",
]

MODEL_SPEC_COLS = [
    "run_hash",
    "dataset_key",
    "scene_name",
    "variant_name",
    "variant_index",
    "matrix_preset",
    *ABLATION_BOOL_COLS,
    *CHECKPOINT_COLS,
]

ABLATION_COLS = [
    "scene_name",
    *ABLATION_BOOL_COLS,
    "eval_checkpoint_iteration",
]

RADAR_SPECS = {
    "+PSNR": {"col": "psnr", "higher_is_better": True},
    "+SSIM": {"col": "ssim", "higher_is_better": True},
    "-LPIPS": {"col": "lpips", "higher_is_better": False},
    "+FPS": {"col": "render_fps", "higher_is_better": True},
    "-VRAM": {"col": "peak_eval_vram_mb", "higher_is_better": False},
    "-Size": {"col": "checkpoint_size_mb", "higher_is_better": False},
    "-Time": {"col": "training_wall_clock_sec_at_checkpoint", "higher_is_better": False},
    "-Gaussians": {"col": "final_gaussian_count", "higher_is_better": False},
}

METRIC_COLS = [spec["col"] for spec in RADAR_SPECS.values()]

BOOLEAN_LABEL_PAIRS = {
    "isotropy": ("isotropy", "anisotropy"),
    "appearance": ("appearance", "no appearance"),
    "sorting": ("sorting", "no sorting"),
    "pruning": ("pruning", "no pruning"),
    "dropout": ("dropout", "no dropout"),
    "ess": ("ess", "no ess"),
    "use_usplat": ("usplat", "no usplat"),
    "uncertain": ("uncertain", "no uncertain"),
}

BOOLEAN_LABEL_MAP = {
    col: {
        True: true_label,
        False: false_label,
    }
    for col, (true_label, false_label) in BOOLEAN_LABEL_PAIRS.items()
}


dataset_results = read_all_dataset_results()
apply_dataset_overrides(dataset_results)

final_df = merge_dataset_results(
    dataset_results,
    drop_rules=DROP_RULES,
)

keep_cols = [c for c in MODEL_SPEC_COLS + METRIC_COLS if c in final_df.columns]
print("Keep:", keep_cols)

df_clean = final_df[keep_cols].copy()

print("\nShape: ", df_clean.shape)

In [ ]:
def prt(df):
    print(df.to_string())

def filter_df(df: pd.DataFrame, **filters) -> pd.DataFrame:
    out = df

    for col, value in filters.items():
        if col not in df:
            raise KeyError(f"Missing column: {col}")

        values = value if isinstance(value, (list, tuple, set)) else [value]
        valid = sorted(out[col].dropna().unique())

        out = out[out[col].isin(values)]

        if out.empty:
            print(
                f"{col} looking for {values}. \n"
                f"Valid options: {valid}"
            )
    prt(out)
    return out.copy()

In [ ]:
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

def smooth_line_numpy(xs, ys, points=200, window=3):
    """
    Lightweight smoothing without scipy.
    Uses linear interpolation + moving average without endpoint drop.
    """
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    valid = np.isfinite(xs) & np.isfinite(ys)
    xs = xs[valid]
    ys = ys[valid]

    if len(xs) < 3:
        return xs, ys

    order = np.argsort(xs)
    xs = xs[order]
    ys = ys[order]

    x_smooth = np.linspace(xs.min(), xs.max(), points)
    y_interp = np.interp(x_smooth, xs, ys)

    window = max(1, int(window))
    if window <= 1:
        return x_smooth, y_interp

    kernel = np.ones(window) / window

    pad_left = window // 2
    pad_right = window - 1 - pad_left

    y_padded = np.pad(
        y_interp,
        pad_width=(pad_left, pad_right),
        mode="edge",
    )

    y_smooth = np.convolve(y_padded, kernel, mode="valid")

    y_smooth[0] = y_interp[0]
    y_smooth[-1] = y_interp[-1]

    return x_smooth, y_smooth


def plot_training_1x3(
    df,
    x_col="eval_checkpoint_iteration",
    run_col="run_hash",
    plot_specs=None,
    dataset_name="",
):
    """
    plot_specs: list of dicts, one per subplot row. Each dict:
      metric      – str, column name for y values
      group_cols  – tuple of 2 str column names that form the group key
      color_map   – dict mapping (val1, val2) -> hex color
      ylabel      – str for y-axis label
      invert      – bool, invert y-axis (default False)
      y_scale     – float divisor applied to y values (default 1.0)
    """
    if plot_specs is None:
        plot_specs = []

    def norm(v):
        return str(v).strip().lower().replace("-", "_").replace(" ", "_")

    fig, axes = plt.subplots(len(plot_specs), 1, figsize=(8, 9), sharex=True)
    if len(plot_specs) == 1:
        axes = [axes]

    fig.subplots_adjust(left=0.09, right=0.68, bottom=0.07, top=0.93, hspace=0.15)

    for ax, spec in zip(axes, plot_specs):
        metric     = spec["metric"]
        group_cols = spec["group_cols"]
        color_map  = spec["color_map"]
        ylabel     = spec.get("ylabel", metric.upper())
        invert     = spec.get("invert", False)
        y_scale    = spec.get("y_scale", 1.0)

        if metric not in df.columns or any(c not in df.columns for c in group_cols):
            ax.set_visible(False)
            continue

        skipped = set()

        for _, run_df in df.groupby(run_col):
            run_df = run_df.sort_values(x_col)
            first_valid = run_df.dropna(subset=list(group_cols))
            if first_valid.empty:
                continue

            raw_key  = tuple(first_valid[c].iloc[0] for c in group_cols)
            norm_key = tuple(norm(v) for v in raw_key)

            color = next(
                (c for k, c in color_map.items() if tuple(norm(v) for v in k) == norm_key),
                None,
            )
            if color is None:
                skipped.add(norm_key)
                continue

            xs = run_df[x_col].to_numpy(float) / 1000.0
            ys = run_df[metric].to_numpy(float) / y_scale
            mask = np.isfinite(xs) & np.isfinite(ys)
            xs, ys = xs[mask], ys[mask]
            if len(xs) == 0:
                continue

            x_s, y_s = smooth_line_numpy(xs, ys)
            ax.plot(x_s, y_s, color=color, lw=2)
            ax.scatter(xs, ys, s=18, color=color)

        if skipped:
            print(f"Skipped unrecognized groups for {metric}: {skipped}")

        if invert:
            ax.invert_yaxis()

        ax.set_ylabel(ylabel, fontsize=20, fontweight="bold")
        ax.tick_params(labelsize=12)
        ax.grid(alpha=0.3)
        ax.margins(x=0, y=0.03)

        ax.legend(
            handles=[
                Line2D([0], [0], color=c, lw=2, label=" / ".join(str(v) for v in k))
                for k, c in color_map.items()
            ],
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            fontsize=12,
            frameon=True,
            borderaxespad=0.0,
        )

    axes[-1].set_xlabel(r"Checkpoint iteration, $n \times 10^3$", fontsize=16)
    plot_center_x = (fig.subplotpars.left + fig.subplotpars.right) / 2
    fig.suptitle(dataset_name, fontsize=24, fontweight="bold", x=plot_center_x, y=0.995)
    plt.show()
    


# Metrics

In [ ]:
df_clean = df_clean[df_clean["dataset_key"] == "ablations_fixed_longer/trex"]

In [ ]:
df_clean[df_clean["dataset_key"] == "ablations_fixed_longer/trex"]["sorting"].value_counts()

## Visual

In [ ]:
plot_training_1x3(
    filter_df(
        df_clean,
        scene_name="trex",
        use_usplat=False,
    ),
    dataset_name="trex",
    plot_specs=[
        {
            "metric":     "psnr",
            "group_cols": ("sorting", "dropout"),
            "color_map": {
                ("sort",      "no_dropout"):  "#0b3c8c",
                ("sort",      "yes_dropout"): "#6fa8dc",
                ("sort_free", "no_dropout"):  "#4f0f0f",
                ("sort_free", "yes_dropout"): "#e06666",
            },
            "ylabel": "PSNR",
        },
        {
            "metric":     "lpips",
            "group_cols": ("sorting", "dropout"),
            "color_map": {
                ("sort",      "no_dropout"):  "#0b3c8c",
                ("sort",      "yes_dropout"): "#6fa8dc",
                ("sort_free", "no_dropout"):  "#4f0f0f",
                ("sort_free", "yes_dropout"): "#e06666",
            },
            "ylabel": "LPIPS",
            "invert": False,
        },
        {
            "metric":     "ssim",
            "group_cols": ("sorting", "dropout"),
            "color_map": {
                ("sort",      "no_dropout"):  "#0b3c8c",
                ("sort",      "yes_dropout"): "#6fa8dc",
                ("sort_free", "no_dropout"):  "#4f0f0f",
                ("sort_free", "yes_dropout"): "#e06666",
            },
            "ylabel": "SSIM",
        },
    ],
)

In [ ]:
# lower band 

filtered = filter_df(
    df_clean,
    scene_name="trex",
    use_usplat=False,
)

prt(
    filtered[
        (filtered["checkpoint_name_iteration"] == 4000) &
        (filtered["lpips"] > 0.3)
    ].sort_values("lpips", ascending=False)[ABLATION_COLS]
)

## Memory

In [ ]:
# no_pruning
plot_training_1x3(
    filter_df(df_clean, scene_name="trex", pruning="no_pruning"),
    dataset_name="trex — no_pruning",
    plot_specs=[
        {
            "metric":     "peak_eval_vram_mb",
            "group_cols": ("appearance", "pruning"),
            "color_map": {
                ("rgb", "no_pruning"): "#00c853",
                ("sh3", "no_pruning"): "#b00020",
            },
            "ylabel": "VRAM",
            "invert": True,
        },
        {
            "metric":     "checkpoint_size_mb",
            "group_cols": ("sorting", "appearance"),
            "color_map": {
                ("sort_free", "rgb"): "#ff1744",
                ("sort_free", "sh3"): "#b00020",
                ("sort",      "rgb"): "#00c853",
                ("sort",      "sh3"): "#234aaa",
            },
            "ylabel": "SIZE",
            "invert": True,
        },
        {
            "metric":     "final_gaussian_count",
            "group_cols": ("pruning", "appearance"),
            "color_map": {
                ("no_pruning", "rgb"): "#00c853",
                ("no_pruning", "sh3"): "#234aaa",
            },
            "ylabel": r"Gaussians $\times\,10^4$",
            "invert":  True,
            "y_scale": 10_000.0,
        },
    ],
)

# interleaved_prune_densify
plot_training_1x3(
    filter_df(df_clean, scene_name="trex", pruning="interleaved_prune_densify"),
    dataset_name="trex — interleaved_prune_densify",
    plot_specs=[
        {
            "metric":     "peak_eval_vram_mb",
            "group_cols": ("appearance", "pruning"),
            "color_map": {
                ("rgb", "interleaved"): "#0047ff",
                ("sh3", "interleaved"): "#ff1744",
            },
            "ylabel": "VRAM",
            "invert": True,
        },
        {
            "metric":     "checkpoint_size_mb",
            "group_cols": ("sorting", "appearance"),
            "color_map": {
                ("sort_free", "rgb"): "#ff1744",
                ("sort_free", "sh3"): "#b00020",
                ("sort",      "rgb"): "#00c853",
                ("sort",      "sh3"): "#234aaa",
            },
            "ylabel": "SIZE",
            "invert": True,
        },
        {
            "metric":     "final_gaussian_count",
            "group_cols": ("pruning", "appearance"),
            "color_map": {
                ("interleaved_prune_densify", "rgb"): "#ff1744",
                ("interleaved_prune_densify", "sh3"): "#b00020",
            },
            "ylabel": r"Gaussians $\times\,10^4$",
            "invert":  True,
            "y_scale": 10_000.0,
        },
    ],
)

In [ ]:
filter_df(df_clean, scene_name="trex", pruning="interleaved_prune_densify")["peak_eval_vram_mb"]

In [ ]:
df_clean["pruning"].unique()

## Framerate

df_clean["]

# Ablations

## Sorting

## Appearance

## Pruning

## ESS

## Dropout